In [1]:
# Clone the repository
!git clone https://github.com/rednote-hilab/dots.ocr.git

%cd dots.ocr

fatal: destination path 'dots.ocr' already exists and is not an empty directory.
/home/ubuntu/pdf-parser/dots.ocr


In [3]:
# Delete flash-attn from requirements
!sed -i '/flash-attn/d' requirements.txt

In [2]:
# Install dependencies
!pip3 install torch torchvision torchaudio
!pip3 install -e .

/bin/bash: line 1: pip3: command not found
/bin/bash: line 1: pip3: command not found


In [ ]:
# Download dots.ocr model
!python3 tools/download_model.py

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor
from qwen_vl_utils import process_vision_info

# --- Load the model with CPU-specific configurations ---
model_path = "/home/ubuntu/pdf-parser/dots.ocr/weights/DotsMOCR"
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    attn_implementation="sdpa",  # Change 1: Use "sdpa" instead of "flash_attention_2"
    #torch_dtype=torch.bfloat16,
    dtype=torch.bfloat16,
    device_map="cpu",            # Change 2: Explicitly set the device to "cpu"
    trust_remote_code=True
)
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)

# --- Prepare your image and prompt ---
image_path = "./imgs/b77b188c-1.png" # Or any other image path
prompt = """Please output the layout information from the PDF image, including each layout element's bbox, its category, and the corresponding text content within the bbox.

1. Bbox format: [x1, y1, x2, y2]

2. Layout Categories: The possible categories are ['Caption', 'Footnote', 'Formula', 'List-item', 'Page-footer', 'Page-header', 'Picture', 'Section-header', 'Table', 'Text', 'Title'].

3. Text Extraction & Formatting Rules:
    - Picture: For the 'Picture' category, the text field should be omitted.
    - Formula: Format its text as LaTeX.
    - Table: Format its text as HTML.
    - All Others (Text, Title, etc.): Format their text as Markdown.

4. Constraints:
    - The output text must be the original text from the image, with no translation.
    - All layout elements must be sorted according to human reading order.

5. Final Output: The entire output must be a single JSON object.
""" 

messages = [
    {
        "role": "user",
        "content": [
            { "type": "image", "image": image_path },
            {"type": "text", "text": prompt}
        ]
    }
]

# --- Standard processing steps ---
text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)

# --- Inference ---
generated_ids = model.generate(**inputs, max_new_tokens=2048)
generated_ids_trimmed = [
    out_ids[len(in_ids):]
    for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)
print(output_text)

In [8]:
# Download dots.ocr model
!python3 tools/download_model-4bit.py

Attention: The model save dir dots.mocr should be replace by a name without `.` like DotsMOCR, util we merge our code to transformers.
/home/ubuntu/pdf-parser/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/ubuntu/pdf-parser/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Fetching 17 files:   0%|                                 | 0/17 [00:00<?, ?it/

In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor
from qwen_vl_utils import process_vision_info

# --- Load the model with CPU-specific configurations ---
MODEL_PATH = "/home/ubuntu/pdf-parser/dots.ocr/weights/DotsMOCR"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    attn_implementation="sdpa",  # Change 1: Use "sdpa" instead of "flash_attention_2"
    #torch_dtype=torch.bfloat16,
    dtype=torch.bfloat16,
    device_map="cpu",            # Change 2: Explicitly set the device to "cpu"
    trust_remote_code=True
)
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True, use_fast=True)


# --- Prepare your image and prompt ---
image_path = "./imgs/b77b188c-1.png" # Or any other image path
prompt = """Please output the layout information from the PDF image, including each layout element's bbox, its category, and the corresponding text content within the bbox.

1. Bbox format: [x1, y1, x2, y2]

2. Layout Categories: The possible categories are ['Caption', 'Footnote', 'Formula', 'List-item', 'Page-footer', 'Page-header', 'Picture', 'Section-header', 'Table', 'Text', 'Title'].

3. Text Extraction & Formatting Rules:
    - Picture: For the 'Picture' category, the text field should be omitted.
    - Formula: Format its text as LaTeX.
    - Table: Format its text as HTML.
    - All Others (Text, Title, etc.): Format their text as Markdown.

4. Constraints:
    - The output text must be the original text from the image, with no translation.
    - All layout elements must be sorted according to human reading order.

5. Final Output: The entire output must be a single JSON object.
""" 

messages = [
    {
        "role": "user",
        "content": [
            { "type": "image", "image": image_path },
            {"type": "text", "text": prompt}
        ]
    }
]

# --- Standard processing steps ---
text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)

ModuleNotFoundError: No module named 'flash_attn'

In [ ]:

# --- Inference ---
generated_ids = model.generate(**inputs, max_new_tokens=2048)
generated_ids_trimmed = [
    out_ids[len(in_ids):]
    for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)
print(output_text)